In [0]:
## CELL 1 — Setup
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import broadcast

CATALOG  = "hive_metastore"
YOUR_DB  = "team2_edulearn"
S3_BASE  = 's3://s3-de-q1-26/DE-Training/team2_edulearn_datalake/'

print(f"[EduLearn] Silver Transformation | DB: {YOUR_DB}")

In [0]:
## CELL 2 — Read enrollments from Bronze, apply transformations
df_bronze = spark.read.table(f"{YOUR_DB}.enrollments_bronze")

print(f"Bronze row count: {df_bronze.count()}")

df_silver_prep = (
    df_bronze
    # Rule 1: Drop null enrollment_id (primary key must exist)
    .filter(F.col("enrollment_id").isNotNull())
    # Rule 2: Drop null total_fees (needed for revenue calculations)
    .filter(F.col("total_fees").isNotNull())
    # Rule 3: Keep only COMPLETED enrollments for performance analysis
    .filter(F.upper(F.col("enrollment_status")) == "COMPLETED")
    # Rule 4: Cast order_date to DateType
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    # Rule 5: Cast total_fees to double
    .withColumn("total_fees", F.col("total_fees").cast("double"))
    # Rule 6: Cast completion_days to int
    .withColumn("completion_days", F.col("completion_days").cast("int"))
    # Rule 7: Standardise enrollment_status to uppercase
    .withColumn("enrollment_status", F.upper(F.col("enrollment_status")))
    # Rule 8: Drop duplicates on enrollment_id
    .dropDuplicates(["enrollment_id"])
    # Rule 9: Add processing_date
    .withColumn("processing_date", F.current_date())
)

print(f"Silver (cleaned) row count: {df_silver_prep.count()}")
print("Row count dropped:", df_bronze.count() - df_silver_prep.count())

In [0]:
## CELL 3 — MERGE into enrollments_silver (idempotent upsert)
silver_path = f"{S3_BASE}/silver/enrollments/"

# Create Silver table if it doesn't exist yet
if not DeltaTable.isDeltaTable(spark, silver_path):
    df_silver_prep.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.enrollments_silver
        USING DELTA LOCATION '{silver_path}'
    """)
    print("[EduLearn] enrollments_silver created for the first time")
else:
    dt_silver = DeltaTable.forPath(spark, silver_path)
    dt_silver.alias("tgt").merge(
        df_silver_prep.alias("src"),
        "tgt.enrollment_id = src.enrollment_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("[EduLearn] enrollments_silver MERGE complete (idempotent upsert)")

spark.sql(f"SELECT COUNT(*) AS silver_count FROM {YOUR_DB}.enrollments_silver").show()

In [0]:
## CELL 4 — customers_silver
df_cust_bronze = spark.read.table(f"{YOUR_DB}.customers_bronze")

df_cust_silver = (
    df_cust_bronze
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
    .withColumn("city", F.initcap(F.col("city")))
    .withColumn("processing_date", F.current_date())
)

silver_cust_path = f"{S3_BASE}/silver/customers/"

if not DeltaTable.isDeltaTable(spark, silver_cust_path):
    df_cust_silver.write.format("delta").mode("overwrite").save(silver_cust_path)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.customers_silver
        USING DELTA LOCATION '{silver_cust_path}'
    """)
else:
    dt_cust = DeltaTable.forPath(spark, silver_cust_path)
    dt_cust.alias("tgt").merge(
        df_cust_silver.alias("src"),
        "tgt.customer_id = src.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print(f"[EduLearn] customers_silver count: "
      f"{spark.sql(f'SELECT COUNT(*) FROM {YOUR_DB}.customers_silver').collect()[0][0]}")

In [0]:
## CELL 5 — courses_silver, enrollment_items_silver, completions_silver
# courses_silver
df_courses_bronze = spark.read.table(f"{YOUR_DB}.courses_bronze")
df_courses_silver = (
    df_courses_bronze
    .filter(F.col("course_id").isNotNull())
    .dropDuplicates(["course_id"])
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("processing_date", F.current_date())
)


courses_path = f"{S3_BASE}/silver/courses/"

if not DeltaTable.isDeltaTable(spark, courses_path):
    df_courses_silver.write.format("delta").mode("overwrite").save(courses_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.courses_silver
        USING DELTA LOCATION '{courses_path}'
    """)
else:
    dt_courses = DeltaTable.forPath(spark, courses_path)

    dt_courses.alias("tgt").merge(
        df_courses_silver.alias("src"),
        "tgt.course_id = src.course_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print(f"[EduLearn] courses_silver count: "
      f"{spark.sql(f'SELECT COUNT(*) FROM {YOUR_DB}.courses_silver').collect()[0][0]}")

# enrollment_items_silver
items_path = f"{S3_BASE}/silver/enrollment_items/"

df_items_bronze = spark.read.table(f"{YOUR_DB}.enrollment_items_bronze")

df_items_silver = (
    df_items_bronze
    .filter(F.col("item_id").isNotNull())
    .dropDuplicates(["item_id"])
    .withColumn("fee", F.col("fee").cast("double"))
    .withColumn("final_fee", F.col("final_fee").cast("double"))
    .withColumn("discount_pct", F.col("discount_pct").cast("double"))
    .withColumn("processing_date", F.current_date())
)

if not DeltaTable.isDeltaTable(spark, items_path):
    df_items_silver.write.format("delta").mode("overwrite").save(items_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.enrollment_items_silver
        USING DELTA LOCATION '{items_path}'
    """)
else:
    dt_items = DeltaTable.forPath(spark, items_path)

    dt_items.alias("tgt").merge(
        df_items_silver.alias("src"),
        "tgt.item_id = src.item_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print(f"[EduLearn] enrollment_items_silver count: "
      f"{spark.sql(f'SELECT COUNT(*) FROM {YOUR_DB}.enrollment_items_silver').collect()[0][0]}")

# completions_silver
comp_path = f"{S3_BASE}/silver/completions/"

df_comp_bronze = spark.read.table(f"{YOUR_DB}.completions_bronze")

df_comp_silver = (
    df_comp_bronze
    .filter(F.col("completion_id").isNotNull())
    .dropDuplicates(["completion_id"])
    .withColumn("score_pct", F.col("score_pct").cast("double"))
    .withColumn("rating", F.col("rating").cast("double"))
    .withColumn("completion_days", F.col("completion_days").cast("int"))
    .withColumn("processing_date", F.current_date())
)

if not DeltaTable.isDeltaTable(spark, comp_path):
    df_comp_silver.write.format("delta").mode("overwrite").save(comp_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.completions_silver
        USING DELTA LOCATION '{comp_path}'
    """)
else:
    dt_comp = DeltaTable.forPath(spark, comp_path)

    dt_comp.alias("tgt").merge(
        df_comp_silver.alias("src"),
        "tgt.completion_id = src.completion_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print(f"[EduLearn] completions_silver count: "
      f"{spark.sql(f'SELECT COUNT(*) FROM {YOUR_DB}.completions_silver').collect()[0][0]}")

print("[EduLearn] All Silver tables created")

In [0]:
## CELL 6 — Enable Change Data Feed on enrollments_silver
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.enrollments_silver
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("[EduLearn] Change Data Feed enabled on enrollments_silver")

In [0]:
## CELL 7 — SQL Task S3: CTE + Window Functions

# S3a: Running fee revenue total per city by month
spark.sql(f"""
    WITH monthly_revenue AS (
        SELECT
            city,
            DATE_TRUNC('month', order_date) AS month,
            ROUND(SUM(total_fees), 2) AS monthly_revenue
        FROM {YOUR_DB}.enrollments_silver
        WHERE order_date IS NOT NULL AND total_fees IS NOT NULL
        GROUP BY city, DATE_TRUNC('month', order_date)
    )
    SELECT
        city,
        month,
        monthly_revenue,
        ROUND(SUM(monthly_revenue) OVER (
            PARTITION BY city
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) AS running_total
    FROM monthly_revenue
    ORDER BY city, month
""").show(50)

# S3b: Top 3 learners per city by enrollment count
spark.sql(f"""
    WITH customer_counts AS (
        SELECT
            customer_id,
            city,
            COUNT(*) AS enrollment_count
        FROM {YOUR_DB}.enrollments_silver
        GROUP BY customer_id, city
    ),
    ranked AS (
        SELECT
            customer_id,
            city,
            enrollment_count,
            RANK() OVER (
                PARTITION BY city
                ORDER BY enrollment_count DESC
            ) AS city_rank
        FROM customer_counts
    )
    SELECT customer_id, city, enrollment_count, city_rank
    FROM ranked
    WHERE city_rank <= 3
    ORDER BY city, city_rank
""").show()

In [0]:
## CELL 8 — Spark Task SP2: Broadcast Join
courses_df = spark.read.table(f"{YOUR_DB}.courses_silver")
items_df   = spark.read.table(f"{YOUR_DB}.enrollment_items_silver")

result = items_df.join(broadcast(courses_df), 'course_id', 'left')
result.explain(True)

# Paste the explain output into SKILLS_EVIDENCE.md under Q9